## RAG Piplines - Data Ingestion toVector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Administrator\AppData\Local\Temp\ipykernel_20204\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            # actual Extraction starts
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF files to process

Processing: High Court.pdf
  ✓ Loaded 29 pages

Processing: LPIV STQA FINAL (1).pdf
  ✓ Loaded 18 pages

Processing: NIPS-2017-attention-is-all-you-need-Paper.pdf
  ✓ Loaded 11 pages

Total documents loaded: 58


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'FPDF 1.86', 'creator': 'PyPDF', 'creationdate': '2026-08-02T20:46:02+05:30', 'source': '..\\data\\pdf\\High Court.pdf', 'total_pages': 29, 'page': 0, 'page_label': '1', 'source_file': 'High Court.pdf', 'file_type': 'pdf'}, page_content="BOMBAY HIGHCOURT\nDAILY MAIN CAUSELIST\nFor Monday The 3rd August 2026\nCOURT NO. 10\nSINGLE\nAT 11:00 AM\nHON'BLE SHRI JUSTICE SHIVKUMAR DIGE\nHEADER NOTE : \n1) Link : - https://hcbombay.vconsol.com/\n2) Passing Order matters From Sr No. 2 to 11 Will be Called out at 03:00 PM.\nSr. CASE NO. PARTIES ADVOCATES\nFor extension of time\n1 WP/4363/2025\n[Criminal]\nSHRUTI SWAPNIL AHER ALIAS\nSHRUTI SHAM KANDEKAR\nVS\nSWAPNIL SAMPATRAO AHER AND\nANR\nBalasaheb Deshmukh\n \nREMARK : (For Extension of time) # Note:- 1) Cri. Writ Petition No. 4363/2025 is disposed of on 12/08/2025. 2)\nWrit issued on 26/08/2025. 3) Now letter dated 30/06/2026 is received on 03/07/2026 from Smt. S. R. Badve,\n09th Jt. C.J.J.D. and J.M.F.C., Crimi

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    #actual splitting operation.
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Split 58 documents into 204 chunks

Example chunk:
Content: BOMBAY HIGHCOURT
DAILY MAIN CAUSELIST
For Monday The 3rd August 2026
COURT NO. 10
SINGLE
AT 11:00 AM
HON'BLE SHRI JUSTICE SHIVKUMAR DIGE
HEADER NOTE : 
1) Link : - https://hcbombay.vconsol.com/
2) Pas...
Metadata: {'producer': 'FPDF 1.86', 'creator': 'PyPDF', 'creationdate': '2026-08-02T20:46:02+05:30', 'source': '..\\data\\pdf\\High Court.pdf', 'total_pages': 29, 'page': 0, 'page_label': '1', 'source_file': 'High Court.pdf', 'file_type': 'pdf'}


### Embeddings and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Administrator\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        #all-MiniLM-L6-v2 is a pretrained sentence embedding model available through Hugging Face.
        # The model has already learned to convert text into meaningful vector representations during its previous training.
        # We are not training it from scratch.
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            #actual model-loading operation.
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        #encode function convert input text into embedding

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
        #returns numpy array


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1137.78it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Administrator\AppData\Local\Temp\ipykernel_20204\1075853731.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 161


In [9]:
# Extract text from every chunk
texts = [doc.page_content for doc in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Check embeddings
print(embeddings.shape)

print(embeddings[0])



Generating embeddings for 204 texts...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches: 100%|██████████| 7/7 [00:17<00:00,  2.51s/it]

Generated embeddings with shape: (204, 384)
(204, 384)
[-6.08492419e-02  1.15961870e-02  3.39837298e-02 -1.11442186e-01
 -1.95211805e-02  7.01156780e-02 -4.47907411e-02  5.86750135e-02
  6.00867299e-03  4.08215337e-02  4.50326763e-02  3.10158636e-02
 -4.09671571e-03 -3.05797383e-02  1.05028842e-02 -7.42648961e-03
 -3.63174151e-03 -8.97374656e-03 -4.79452945e-02 -6.83761295e-03
  7.12701259e-03  5.37717678e-02 -3.07878517e-02 -2.25008018e-02
 -5.11734746e-02  3.04124947e-03  3.18053993e-03 -2.14528590e-02
 -1.76028349e-02 -3.47901210e-02  1.13735506e-02  1.16951153e-01
 -7.18468577e-02 -2.08073482e-02  2.06927396e-02 -4.57475223e-02
 -6.27805805e-03 -2.32104622e-02  4.99625467e-02 -4.79081906e-02
  7.37606212e-02 -4.69713286e-02 -2.11744905e-02  2.79276632e-02
  1.93842854e-02 -1.82762612e-02 -1.81082636e-02  6.71009272e-02
 -6.49515763e-02  1.22420685e-02 -1.02103740e-01 -3.49516124e-02
  6.78951060e-03  1.69571862e-02 -1.87034893e-03 -8.56615752e-02
  3.93709391e-02 -6.75874576e-03  3

In [10]:
## store in vector database
vectorstore.add_documents(chunks,embeddings)

Adding 204 documents to vector store...
Successfully added 204 documents to vector store
Total documents in collection: 365


## Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -1.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Who is the judge in Court No. 10?")

Retrieving documents for query: 'Who is the judge in Court No. 10?'
Top K: 5, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.13it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ac3745dd_38',
  'content': "DAILY MAIN CAUSELIST COURT NO 10-FOR MONDAY THE 3RD AUGUST 2026  BOMBAY HIGHCOURT - 10 - \n   WITH  \nABA/1542/2026 DINKAR SAKHARAM BODKE MANOJ S PINGALE\n[Criminal] VS Akshay Karlekar\nSTATE OF MAHARASHTRA Rohit D Gorade\nREMARK: (For Anticipatory Bail) Note : 1. Application does not accompany certified copy of judgment or\norder dtd. 08/06/2026 of trial Court and First Appellate Court, if any. UT accepted by RJ II. 2. Copy Of\nOriginal FIR is not filed. UT accepted by RJ II. FIR No. 67 of 2026 UT given. 3.Amendment carried out\nforthwith by the Advocate for the petitioner in Court as per Court's order dated 19/06/2026. 4. As per\nCourt's order dated 24/06/2026, the first informant was present in the Court, but he has not engaged any\nAdvocate till today to represent him. Court S. J. for Order.\n35 ABA/1602/2026\n[Criminal]\nHARAN KALYAN CHAVAN\nVS\nTHE STATE OF MAHARASHTRA\nPRASANNA KISHOR SHAHANE\n \nREMARK : (For ABA) Note: 1) Application do

### Integeration of Vectordb context pipeline with LLM output

In [17]:
### Simple RAG Pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

### Initialize the Groq LLM

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)


## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):

    ## Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    ## Generate the answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely.

    Context:
    {context}

    Question: {query}

    Answer:"""

    response = llm.invoke([prompt])

    return response.content

In [19]:
answer = rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
An attention mechanism is a way for a model to focus on different parts of an input sequence when producing each output. It computes a **query** vector for the current position, compares it to **key** vectors from all positions (usually via a dot‑product), scales the resulting scores, and turns them into a probability distribution with a softmax. These weights are then used to take a weighted sum of **value** vectors, yielding a context‑aware representation that highlights the most relevant information for that position. Multi‑head attention extends this by performing several such projections in parallel, allowing the model to capture different types of relationships simultaneously.


In [20]:
answer = rag_simple("What is STQA?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is STQA?'
Top K: 3, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**STQA** stands for **Software Testing and Quality Assurance**. It is the discipline that plans, executes, and manages testing activities to verify that software meets its requirements and quality standards, ensuring reliability, performance, and user satisfaction.


## Enhanced RAG Pipeline Features

In [23]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is attention mechanism", rag_retriever, llm, top_k=3, min_score=-1.0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is attention mechanism'
Top K: 3, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: An attention mechanism lets a model weigh different parts of its input when computing each output. In practice, it projects queries, keys, and values into vectors, computes dot‑products between queries and keys to get scores, scales and softmaxes them to obtain attention weights, and then forms a weighted sum of the values. Multi‑head attention repeats this in parallel on several sub‑spaces, allowing the model to capture diverse relationships. The result is a context‑aware representation that can be used for tasks like translation, summarization, or classification.
Sources: [{'source': 'NIPS-2017-attention-is-all-you-need-Paper.pdf', 'page': 3, 'score': -0.07149481773376465, 'preview': 'we found it beneﬁcial to linearly project the queries, keys and values h times with different, learned\nlinear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of\nqueries, keys and values we then perform the attention function in parallel, yielding dv -

## Advanced RAG Pipeline: Streaming, Citations, History, Summarization--

In [25]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=-1.0, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.32it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
convolution is equal to the combination of a self-attention layer and a point-wise feed-for

ward layer,
the approach we take in our model.
As side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions
from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic
and semantic structure of the sentences.
5 Training
This section describes the training regime for our models.
5.1 Training Data and Batching
We trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million
sentence pairs. Sentences were encoded using byte-pair encoding [3], which has a shared source-
target vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT
2014 English-French dataset consisting of 36M sentences and split tokens into a 32000 word-piece

Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Att